# Chapter 25 — Transformer I: Attention, Multi-Head Attention & the Encoder-Decoder

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part4_transformers_genai/ch25_transformer_i/ch25_notebook.ipynb)

Companion notebook for **Chapter 25** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- Attention replaces recurrence, so every position in a sequence can be processed in parallel.
- Query, Key and Value are three roles: the query is matched against the keys, and the resulting weights combine the values.
- Dividing by the square root of d keeps softmax out of its saturated region, so gradients stay usable.
- Multiple heads let the model capture several different kinds of relationship at once; W_o then mixes them back together.
- An encoder layer is self-attention plus a position-wise feedforward network, with the feedforward weights shared across all positions.

## Example 1: Scaled Dot-Product Attention from Scratch

This builds the core attention computation exactly as defined in the overview. We build Query, Key and Value vectors, compute the scaled dot products alpha_tm = <q_t,k_m>/sqrt(d), apply softmax to get attention scores, and form the context vector as the weighted sum of Value vectors. The final section shows concretely why the 1/sqrt(d) scaling matters: without it, large dot products saturate the softmax and gradients vanish.

In [1]:
# Chapter 25 - Example 1: Scaled Dot-Product Attention (SDPA) from scratch
import numpy as np
np.random.seed(0)

def softmax(v):
    e = np.exp(v - v.max())
    return e / e.sum()

def sdpa(q, K, V, d):
    """q: query (d,) | K: keys (T,d) | V: values (T,d)"""
    alpha = (K @ q) / np.sqrt(d)      # scaled dot products
    a     = softmax(alpha)            # attention scores
    c     = a @ V                     # context vector
    return alpha, a, c

d, T = 4, 5
K = np.round(np.random.randn(T, d), 2)   # Key vectors
V = np.round(np.random.randn(T, d), 2)   # Value vectors
q = np.round(np.random.randn(d), 2)      # Query vector for position t

print("=== Scaled Dot-Product Attention ===\n")
print("Terminology borrowed from regression: Query, Key, Value\n")
print(f"Query q_t          : {q}")
print(f"Key matrix K  ({T}x{d}):\n{K}")
print(f"Value matrix V ({T}x{d}):\n{V}\n")

alpha, a, c = sdpa(q, K, V, d)

print("Step 1: scaled dot product  alpha_tm = <q_t, k_m> / sqrt(d)")
for m in range(T):
    print(f"  alpha_t{m+1} = {K[m]} . q / sqrt({d}) = {alpha[m]:+.4f}")

print("\nStep 2: attention scores  a_tm = softmax(alpha_tm)")
for m in range(T):
    bar = "#" * int(a[m] * 40)
    print(f"  a_t{m+1} = {a[m]:.4f}  {bar}")
print(f"  Sum of attention scores = {a.sum():.4f}  (softmax always sums to 1)")

print("\nStep 3: context vector  c_t = sum_m a_tm * v_m")
print(f"  c_t = {np.round(c, 4)}")

print("\nInterpretation:")
top = int(np.argmax(a))
print(f"  Key {top+1} receives the highest attention ({a[top]:.4f}),")
print(f"  so the context vector is pulled most toward Value vector {top+1}.")
print("\nWhy divide by sqrt(d)? Without scaling, large dot products push softmax")
print("into a saturated region where gradients vanish during training.")

# Show the saturation effect
print("\n=== Effect of the 1/sqrt(d) scaling ===")
big = np.array([10.0, 8.0, 6.0, 4.0, 2.0])
print(f"  Unscaled logits {big} -> softmax {np.round(softmax(big), 4)}")
print(f"  Scaled by sqrt({d})            -> softmax {np.round(softmax(big/np.sqrt(d)), 4)}")
print("  Scaling keeps the distribution smoother (non-saturated gradients).")

=== Scaled Dot-Product Attention ===

Terminology borrowed from regression: Query, Key, Value

Query q_t          : [-1.05 -1.42 -1.71  1.95]
Key matrix K  (5x4):
[[ 1.76  0.4   0.98  2.24]
 [ 1.87 -0.98  0.95 -0.15]
 [-0.1   0.41  0.14  1.45]
 [ 0.76  0.12  0.44  0.33]
 [ 1.49 -0.21  0.31 -0.85]]
Value matrix V (5x4):
[[-2.55  0.65  0.86 -0.74]
 [ 2.27 -1.45  0.05 -0.19]
 [ 1.53  1.47  0.15  0.38]
 [-0.89 -1.98 -0.35  0.16]
 [ 1.23  1.2  -0.39 -0.3 ]]

Step 1: scaled dot product  alpha_tm = <q_t, k_m> / sqrt(d)
  alpha_t1 = [1.76 0.4  0.98 2.24] . q / sqrt(4) = +0.1381
  alpha_t2 = [ 1.87 -0.98  0.95 -0.15] . q / sqrt(4) = -1.2445
  alpha_t3 = [-0.1   0.41  0.14  1.45] . q / sqrt(4) = +1.0554
  alpha_t4 = [0.76 0.12 0.44 0.33] . q / sqrt(4) = -0.5387
  alpha_t5 = [ 1.49 -0.21  0.31 -0.85] . q / sqrt(4) = -1.7269

Step 2: attention scores  a_tm = softmax(alpha_tm)
  a_t1 = 0.2264  #########
  a_t2 = 0.0568  ##
  a_t3 = 0.5666  ######################
  a_t4 = 0.1151  ####
  a_t5 = 0.035

## Example 2: Self-Attention with Multi-Head Attention

Here queries, keys and values are all generated from the SAME sequence using W_Q, W_K and W_V - that is what makes it self-attention. We first run single-head attention, then multi-head attention with h=4 heads. Notice each head produces a different attention pattern for the same token, and that concatenating h heads of dimension l = d/h reconstructs a d-dimensional vector, which W_o then transforms into the output.

In [2]:
# Chapter 25 - Example 2: Self-Attention with Multi-Head Attention (MHA)
import numpy as np
np.random.seed(1)

def softmax_rows(M):
    e = np.exp(M - M.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

# Sequence X: T d-dimensional feature vectors (e.g. word embeddings)
T, d, h = 4, 8, 4          # 4 tokens, model dim 8, 4 heads
l = d // h                 # per-head dim  l = d/h
X = np.round(np.random.randn(T, d), 2)

print("=== Self-Attention: Single-Head vs Multi-Head ===\n")
print(f"Sequence X: T={T} tokens, model dimension d={d}")
print(f"Heads h={h}  ->  per-head dimension l = d/h = {l}\n")

def head_attention(X, WQ, WK, WV):
    """One attention head. W*: (l x d). Returns context matrix (T x l)."""
    Q = X @ WQ.T          # (T,l)  query vectors  q~_it = W_i^Q x_t
    K = X @ WK.T          # (T,l)  key vectors
    V = X @ WV.T          # (T,l)  value vectors
    scores = softmax_rows((Q @ K.T) / np.sqrt(l))   # (T,T)
    return scores, scores @ V

# ---- Single-head attention (one set of transformation matrices) ----
WQ = np.random.randn(l, d) * 0.3
WK = np.random.randn(l, d) * 0.3
WV = np.random.randn(l, d) * 0.3
scores1, C1 = head_attention(X, WQ, WK, WV)

print("--- Single-Head Attention ---")
print(f"Transformation matrices W_Q, W_K, W_V each of size l x d = {l}x{d}")
print("Attention score matrix (row t = how token t attends to every token):")
print(np.round(scores1, 3))
print(f"Row sums: {np.round(scores1.sum(axis=1), 4)}  (each row sums to 1)")
print(f"Context matrix C~ shape: {C1.shape}  (l x T transposed -> T x l)\n")

# ---- Multi-head attention ----
print("--- Multi-Head Attention ---")
heads, all_scores = [], []
for i in range(h):
    WQi = np.random.randn(l, d) * 0.3
    WKi = np.random.randn(l, d) * 0.3
    WVi = np.random.randn(l, d) * 0.3
    s_i, C_i = head_attention(X, WQi, WKi, WVi)
    heads.append(C_i); all_scores.append(s_i)

print("Each head learns a DIFFERENT attention pattern for token 1:")
for i, s in enumerate(all_scores, 1):
    print(f"  Head {i}: {np.round(s[0], 3)}")

C_concat = np.concatenate(heads, axis=1)    # (T, h*l) = (T, d)
Wo = np.random.randn(d, d) * 0.3            # d x d output matrix
Z  = C_concat @ Wo.T                        # output sequence Z

print(f"\nConcatenated context: {h} heads x {l} dims = {C_concat.shape[1]} = d")
print(f"  c_t = concat(c_1t, ..., c_{h}t)  ->  shape {C_concat.shape}")
print(f"Output z_t = W_o c_t                ->  shape {Z.shape}")
print(f"\nOutput sequence Z (first 2 tokens):")
print(np.round(Z[:2], 4))

print("\nWhy multiple heads? Each head uses its own W_Q/W_K/W_V, so different")
print("heads can capture different relationships (syntax, coreference, etc.)")
print("in parallel. Their outputs are concatenated and mixed by W_o.")

=== Self-Attention: Single-Head vs Multi-Head ===

Sequence X: T=4 tokens, model dimension d=8
Heads h=4  ->  per-head dimension l = d/h = 2

--- Single-Head Attention ---
Transformation matrices W_Q, W_K, W_V each of size l x d = 2x8
Attention score matrix (row t = how token t attends to every token):
[[0.277 0.261 0.216 0.246]
 [0.508 0.234 0.075 0.183]
 [0.131 0.269 0.354 0.246]
 [0.365 0.242 0.162 0.231]]
Row sums: [1. 1. 1. 1.]  (each row sums to 1)
Context matrix C~ shape: (4, 2)  (l x T transposed -> T x l)

--- Multi-Head Attention ---
Each head learns a DIFFERENT attention pattern for token 1:
  Head 1: [0.32  0.325 0.126 0.229]
  Head 2: [0.3   0.289 0.201 0.21 ]
  Head 3: [0.275 0.202 0.25  0.274]
  Head 4: [0.481 0.154 0.169 0.196]

Concatenated context: 4 heads x 2 dims = 8 = d
  c_t = concat(c_1t, ..., c_4t)  ->  shape (4, 8)
Output z_t = W_o c_t                ->  shape (4, 8)

Output sequence Z (first 2 tokens):
[[ 0.329  -0.1075 -0.7072 -0.875  -0.1073  0.6758 -0.3755 

## Example 3: Masked Self-Attention and the PWFFNN

The decoder must not look at future words, so a causal mask sets those attention scores to exactly zero - you can see the upper triangle vanish while every row still sums to 1. The second half builds the Position-Wise Feedforward Network and shows weight sharing: T separate MLFFNNs would need T times the parameters, but sharing the weight matrices across positions cuts that by 80% for T=5.

In [3]:
# Chapter 25 - Example 3: Masked Self-Attention (decoder) + PWFFNN weight sharing
import numpy as np
np.random.seed(2)

def softmax_rows(M):
    e = np.exp(M - M.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

T, d = 5, 6
X = np.random.randn(T, d)

print("=== Masked Self-Attention in the Decoder ===\n")
print("In the decoder, the output word at time t may depend ONLY on words")
print("y_1 ... y_t (already generated). Future words must be hidden -> masking.\n")

Q = np.random.randn(T, d) * 0.3
K = np.random.randn(T, d) * 0.3
V = np.random.randn(T, d) * 0.3
raw = (Q @ K.T) / np.sqrt(d)

# Causal mask: position t cannot look at positions > t
mask = np.triu(np.ones((T, T)), k=1).astype(bool)
print("Causal mask (True = blocked / future position):")
print(mask.astype(int))

masked = raw.copy()
masked[mask] = -np.inf          # -inf -> softmax gives exactly 0

scores_unmasked = softmax_rows(raw)
scores_masked   = softmax_rows(masked)

print("\nAttention scores WITHOUT mask (token 2 can see the future - wrong!):")
print(np.round(scores_unmasked, 3))
print("\nAttention scores WITH causal mask (upper triangle is exactly 0):")
print(np.round(scores_masked, 3))
print(f"\nRow sums with mask: {np.round(scores_masked.sum(axis=1), 4)} (still valid distributions)")
print("Token 1 attends only to itself; token 5 can attend to all 5 positions.")

# ---- PWFFNN: same weights applied at every position ----
print("\n=== Position-Wise Feedforward Neural Network (PWFFNN) ===\n")
Z = scores_masked @ V        # output of masked MHA, shape (T,d)
d_ff = 12
W1 = np.random.randn(d, d_ff) * 0.3;  b1 = np.zeros(d_ff)
W2 = np.random.randn(d_ff, d) * 0.3;  b2 = np.zeros(d)

def pwffnn(z):               # one MLFFNN applied to ONE position
    hidden = np.maximum(0, z @ W1 + b1)     # nonlinear hidden layer
    return hidden @ W2 + b2

out = np.vstack([pwffnn(Z[t]) for t in range(T)])

print(f"There are T={T} MLFFNNs (one per position), but the weight matrices")
print(f"W_FF1 {W1.shape} and W_FF2 {W2.shape} are SHARED across all positions.")
print(f"Parameters if NOT shared : {T} x ({W1.size + W2.size}) = {T*(W1.size+W2.size)}")
print(f"Parameters WITH sharing  : {W1.size + W2.size}")
print(f"Reduction                : {(1 - 1/T)*100:.0f}% fewer parameters\n")
print(f"Input to PWFFNN  (position 1): {np.round(Z[0], 4)}")
print(f"Output of PWFFNN (position 1): {np.round(out[0], 4)}")
print(f"Output sequence shape: {out.shape}  (same length and dimension as input)")

# Verify weight sharing: feeding the same vector at 2 positions gives same output
same = np.allclose(pwffnn(Z[0]), pwffnn(Z[0].copy()))
print(f"\nSame input at any position -> same output? {same}  (confirms shared weights)")

=== Masked Self-Attention in the Decoder ===

In the decoder, the output word at time t may depend ONLY on words
y_1 ... y_t (already generated). Future words must be hidden -> masking.

Causal mask (True = blocked / future position):
[[0 1 1 1 1]
 [0 0 1 1 1]
 [0 0 0 1 1]
 [0 0 0 0 1]
 [0 0 0 0 0]]

Attention scores WITHOUT mask (token 2 can see the future - wrong!):
[[0.209 0.19  0.231 0.197 0.173]
 [0.192 0.202 0.197 0.207 0.202]
 [0.208 0.206 0.192 0.182 0.212]
 [0.205 0.204 0.195 0.189 0.207]
 [0.21  0.188 0.224 0.184 0.194]]

Attention scores WITH causal mask (upper triangle is exactly 0):
[[1.    0.    0.    0.    0.   ]
 [0.487 0.513 0.    0.    0.   ]
 [0.343 0.34  0.317 0.    0.   ]
 [0.258 0.258 0.246 0.238 0.   ]
 [0.21  0.188 0.224 0.184 0.194]]

Row sums with mask: [1. 1. 1. 1. 1.] (still valid distributions)
Token 1 attends only to itself; token 5 can attend to all 5 positions.

=== Position-Wise Feedforward Neural Network (PWFFNN) ===

There are T=5 MLFFNNs (one per pos